# Weather Forecast Modeling V1.1 — Canonical Colab GPU run

This notebook executes the frozen T2H protocol from commit 28d6b569f7ba9a883ce294edde67ce746dc4ee67. The local CUDA run is LOCAL_PRELIMINARY and is not canonical. This run does not redownload provider data, redesign features, change the target/split/candidates, or tune from the known local TEST result.

It transfers the validated 189-partition dataset from MyDrive/weather-streaming-bigdata/datasets/weather_forecast_fe_t2h_v1_1/20261003T070520Z-xgb-t2h-v1-1/, verifies archive SHA-256, Parquet checksums and footers, the dataset manifest and the 73-feature identity, then trains on a verified Colab GPU. TEST values are not materialized before freeze and are evaluated once.

The canonical model, full TEST prediction Parquet, and run evidence are written to the run-specific Drive artifact directory. The local preliminary model and predictions remain separately labeled and preserved.

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive', force_remount=False)
MY_DRIVE = Path('/content/drive/MyDrive')
assert MY_DRIVE.is_dir(), 'Google Drive mount is not available'


In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO_URL = 'https://github.com/HaoLe2710/weather-streaming-bigdata.git'
BRANCH = 'model/weather-forecast-xgboost-t2h-v1'
FROZEN_PROTOCOL_COMMIT = '28d6b569f7ba9a883ce294edde67ce746dc4ee67'
REPO = Path('/content/weather-streaming-bigdata')

if not REPO.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO)], check=True)
else:
    if not (REPO / '.git').exists():
        raise RuntimeError(f'Unexpected path is not a Git checkout: {REPO}')
    dirty = subprocess.check_output(['git', '-C', str(REPO), 'status', '--porcelain'], text=True).strip()
    if dirty:
        raise RuntimeError('Refusing to overwrite a non-clean protocol checkout')
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', BRANCH], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', FROZEN_PROTOCOL_COMMIT], check=True)
GIT_COMMIT = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
assert GIT_COMMIT == FROZEN_PROTOCOL_COMMIT, (GIT_COMMIT, FROZEN_PROTOCOL_COMMIT)
git_remote = subprocess.check_output(['git', '-C', str(REPO), 'remote', 'get-url', 'origin'], text=True).strip()
assert git_remote.rstrip('.git') == REPO_URL.rstrip('.git'), git_remote
assert not subprocess.check_output(['git', '-C', str(REPO), 'status', '--porcelain'], text=True).strip()

subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(REPO / 'requirements-ml.txt')], check=True)
sys.path.insert(0, str(REPO))
os.chdir(REPO)
print({'git_commit': GIT_COMMIT, 'branch': BRANCH, 'repository': str(REPO)})


In [ ]:
import json, platform, re, subprocess, sys
import numpy as np
import pandas as pd
import pyarrow as pa
import sklearn
import xgboost as xgb

try:
    import google.colab
    IN_COLAB = True
except Exception:
    IN_COLAB = False

try:
    nvidia_smi = subprocess.check_output(['nvidia-smi'], text=True, stderr=subprocess.STDOUT)
    gpu_csv = subprocess.check_output(
        ['nvidia-smi', '--query-gpu=name,driver_version,memory.total', '--format=csv,noheader'],
        text=True,
    ).strip()
    gpu_rows = [line.strip() for line in gpu_csv.splitlines() if line.strip()]
except Exception as exc:
    nvidia_smi, gpu_rows, nvidia_smi_error = '', [], repr(exc)
else:
    nvidia_smi_error = None

build_info = xgb.build_info() if hasattr(xgb, 'build_info') else {}
cuda_build_value = build_info.get('USE_CUDA')
cuda_build_enabled = str(cuda_build_value).strip().upper() in {'TRUE', '1', 'ON', 'YES'}
gpu_probe_config = None
gpu_probe_error = None
if gpu_rows and cuda_build_enabled:
    try:
        probe_x = np.arange(4096, dtype=np.float32).reshape(1024, 4) / 4096.0
        probe_y = probe_x[:, 0] + probe_x[:, 1]
        probe_booster = xgb.train(
            {'objective':'reg:squarederror','tree_method':'hist','device':'cuda','max_depth':1,'verbosity':0},
            xgb.DMatrix(probe_x, label=probe_y),
            num_boost_round=1,
        )
        gpu_probe_config = json.loads(probe_booster.save_config())
    except Exception as exc:
        gpu_probe_error = repr(exc)

def _config_uses_cuda(node):
    if isinstance(node, dict):
        for key, value in node.items():
            if key == 'device' and 'cuda' in str(value).lower():
                return True
            if _config_uses_cuda(value):
                return True
    elif isinstance(node, list):
        return any(_config_uses_cuda(value) for value in node)
    return False

cuda_available = bool(gpu_rows) and cuda_build_enabled and _config_uses_cuda(gpu_probe_config)
cuda_match = re.search(r'CUDA Version:\s*([^\s]+)', nvidia_smi)
ENVIRONMENT = {
    'canonical_environment':'GOOGLE_COLAB', 'is_colab':IN_COLAB,
    'python':platform.python_version(), 'python_executable':sys.executable,
    'xgboost':xgb.__version__, 'numpy':np.__version__, 'pandas':pd.__version__,
    'pyarrow':pa.__version__, 'scikit_learn':sklearn.__version__,
    'gpu_rows':gpu_rows, 'gpu_name':gpu_rows[0].split(',')[0].strip() if gpu_rows else None,
    'cuda_available':cuda_available, 'cuda_build_enabled':cuda_build_enabled,
    'xgboost_build_info':build_info, 'xgboost_cuda_probe_passed':_config_uses_cuda(gpu_probe_config),
    'xgboost_cuda_probe_error':gpu_probe_error, 'nvidia_smi_error':nvidia_smi_error,
    'nvidia_smi_cuda_version':cuda_match.group(1) if cuda_match else None,
    'nvidia_smi_header':nvidia_smi.splitlines()[:5],
}
print(json.dumps(ENVIRONMENT, indent=2, default=str))
if not IN_COLAB or not cuda_available:
    raise RuntimeError('STOP_CANONICAL_TRAINING_NO_GPU: actual Google Colab CUDA GPU and XGBoost CUDA probe are required')


In [ ]:
import hashlib, json, shutil, zipfile
from datetime import datetime, timezone
from pathlib import Path

LOCAL_DATASET_VERSION = '20261003T070520Z-xgb-t2h-v1-1'
DRIVE_PROJECT = MY_DRIVE / 'weather-streaming-bigdata'
DRIVE_DATASET_ROOT = DRIVE_PROJECT / 'datasets' / 'weather_forecast_fe_t2h_v1_1' / LOCAL_DATASET_VERSION
ARTIFACT_BASE = DRIVE_PROJECT / 'artifacts' / 'weather_forecast_xgboost_t2h_v1_1'
ARTIFACT_BASE.mkdir(parents=True, exist_ok=True)
RUN_CONTROL_PATH = ARTIFACT_BASE / 'canonical_run_control.json'
if RUN_CONTROL_PATH.exists():
    RUN_CONTROL = json.loads(RUN_CONTROL_PATH.read_text(encoding='utf-8'))
    if RUN_CONTROL.get('protocol_commit') != FROZEN_PROTOCOL_COMMIT:
        raise RuntimeError('Existing canonical run control points to another protocol commit')
    RUN_ID = RUN_CONTROL['run_id']
else:
    RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-xgb-t2h-v1-1-colab'
    if RUN_ID == LOCAL_DATASET_VERSION:
        raise RuntimeError('Colab RUN_ID must be new and distinct from the local preliminary run')
    RUN_CONTROL = {'run_id':RUN_ID,'protocol_commit':FROZEN_PROTOCOL_COMMIT,'branch':BRANCH}
    temp_control = RUN_CONTROL_PATH.with_suffix('.tmp')
    temp_control.write_text(json.dumps(RUN_CONTROL, indent=2) + '\n', encoding='utf-8')
    temp_control.replace(RUN_CONTROL_PATH)
ARTIFACT_ROOT = ARTIFACT_BASE / RUN_ID
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
MODEL_PATH = ARTIFACT_ROOT / 'weather_forecast_xgboost_t2h_v1_1.json'
DATASET_ROOT = Path('/content/weather_forecast_fe_t2h_v1_1') / LOCAL_DATASET_VERSION
DATASET_ROOT.mkdir(parents=True, exist_ok=True)

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda:stream.read(8*1024*1024), b''):
            digest.update(block)
    return digest.hexdigest()

def copy_verified(source, destination):
    source, destination = Path(source), Path(destination)
    if not source.is_file():
        raise FileNotFoundError(source)
    observed_sha = sha256_file(source)
    if destination.exists():
        if sha256_file(destination) != observed_sha:
            raise RuntimeError(f'Existing run evidence differs: {destination}')
    else:
        shutil.copy2(source, destination)
    return observed_sha

transfer = json.loads((DRIVE_DATASET_ROOT/'dataset_transfer_manifest.json').read_text(encoding='utf-8'))
assert transfer['dataset_version'] == LOCAL_DATASET_VERSION
assert transfer['source_git_commit'] == FROZEN_PROTOCOL_COMMIT and transfer['branch'] == BRANCH
assert transfer['total_rows'] == 3312666
assert transfer['rows_by_split'] == {'TRAIN':2207394,'VALIDATION':553392,'TEST':551880}
assert transfer['locations'] == 63 and transfer['feature_count'] == 73
assert transfer['feature_list_sha256'] == '20a5d2fb56d9b7231f4c43b39ad7a833298d76b1bfd0f127b2b251c57e5d7fd2'
assert transfer['target_offset_seconds'] == 7200

for name in ['dataset_manifest.json','feature_list.json','feature_contract.json','source_contract.json','split_validation.json']:
    copy_verified(DRIVE_DATASET_ROOT/name, ARTIFACT_ROOT/name)
assert sha256_file(ARTIFACT_ROOT/'dataset_manifest.json') == transfer['dataset_manifest_sha256']
assert sha256_file(ARTIFACT_ROOT/'source_contract.json') == transfer['source_contract_sha256']
assert sha256_file(ARTIFACT_ROOT/'feature_list.json') == transfer['feature_list_artifact_sha256']
LOCAL_EVIDENCE = REPO / 'results' / 'modeling-t2h' / LOCAL_DATASET_VERSION
for name in ['provider_audit.json','raw_validation.json','feature_validation.json','live_forecast_probe.json']:
    copy_verified(LOCAL_EVIDENCE/name, ARTIFACT_ROOT/name)

manifest = json.loads((ARTIFACT_ROOT/'dataset_manifest.json').read_text(encoding='utf-8'))
feature_contract = json.loads((ARTIFACT_ROOT/'feature_contract.json').read_text(encoding='utf-8'))
source_contract = json.loads((ARTIFACT_ROOT/'source_contract.json').read_text(encoding='utf-8'))
split_validation = json.loads((ARTIFACT_ROOT/'split_validation.json').read_text(encoding='utf-8'))
feature_validation = json.loads((ARTIFACT_ROOT/'feature_validation.json').read_text(encoding='utf-8'))
assert manifest['total_rows'] == 3312666 and manifest['rows_by_split'] == {'TRAIN':2207394,'VALIDATION':553392,'TEST':551880}
assert manifest['feature_count'] == 73 and manifest['feature_list_sha256'] == transfer['feature_list_sha256']
assert manifest['locations_by_split'] == {'TRAIN':63,'VALIDATION':63,'TEST':63}
assert feature_contract['feature_count'] == 73 and feature_contract['feature_list_sha256'] == transfer['feature_list_sha256']
assert source_contract['predictor_source']['model_id'] == 'ecmwf_ifs'
assert source_contract['target_source']['model_id'] == 'era5' and source_contract['canonical_target_contract'] == 'B'
assert split_validation['status'] == 'PASS' and split_validation['duplicate_location_feature_time_keys'] == 0
assert split_validation['target_offset_violations'] == 0 and split_validation['target_location_mismatches'] == 0
assert split_validation['feature_null_nan_inf_rows'] == 0 and split_validation['label_null_nan_inf_rows'] == 0
assert split_validation['locations_by_split'] == {'TRAIN':63,'VALIDATION':63,'TEST':63}
assert feature_validation['status'] == 'PASS' and feature_validation['aggregate_future_leakage_check'] == 'PASS'
assert feature_validation['max_feature_source_time_le_feature_time'] is True
assert feature_validation['target_time_offset_seconds'] == 7200 and feature_validation['target_is_in_model_feature_columns'] is False

archive_reports = []
for archive in transfer['archives']:
    archive_path = DRIVE_DATASET_ROOT / archive['file']
    if archive_path.stat().st_size != archive['bytes'] or sha256_file(archive_path) != archive['sha256']:
        raise RuntimeError(f'Dataset archive identity failed: {archive["file"]}')
    with zipfile.ZipFile(archive_path) as zipped:
        bad_member = zipped.testzip()
        if bad_member:
            raise RuntimeError(f'ZIP CRC failed in {archive["file"]}: {bad_member}')
        root_resolved = DATASET_ROOT.resolve()
        for member in zipped.namelist():
            target = (DATASET_ROOT/member).resolve()
            if root_resolved not in target.parents and target != root_resolved:
                raise RuntimeError(f'Unsafe archive path: {member}')
        zipped.extractall(DATASET_ROOT)
    archive_reports.append({'file':archive['file'],'bytes':archive['bytes'],'sha256':archive['sha256'],'status':'PASS'})

from ml.forecast_t2h_v1_1 import validate_dataset_manifest
dataset_identity = validate_dataset_manifest(DATASET_ROOT, ARTIFACT_ROOT)
assert dataset_identity['status'] == 'PASS', dataset_identity
assert dataset_identity['total_rows'] == 3312666
assert dataset_identity['rows_by_split'] == {'TRAIN':2207394,'VALIDATION':553392,'TEST':551880}
assert dataset_identity['locations_by_split'] == {'TRAIN':63,'VALIDATION':63,'TEST':63}
assert dataset_identity['test_values_read'] is False
DATASET_IDENTITY = {
    'status':'PASS','dataset_version':LOCAL_DATASET_VERSION,
    'dataset_manifest_sha256':transfer['dataset_manifest_sha256'],
    'source_contract_sha256':transfer['source_contract_sha256'],
    'feature_list_sha256':transfer['feature_list_sha256'],
    'total_rows':dataset_identity['total_rows'],'rows_by_split':dataset_identity['rows_by_split'],
    'locations_by_split':dataset_identity['locations_by_split'],'feature_count':manifest['feature_count'],
    'parquet_partition_count':len(manifest['partitions']),'target_offset_seconds':7200,
    'duplicates':split_validation['duplicate_location_feature_time_keys'],
    'target_offset_violations':split_validation['target_offset_violations'],
    'target_location_mismatches':split_validation['target_location_mismatches'],
    'feature_null_nan_inf_rows':split_validation['feature_null_nan_inf_rows'],
    'label_null_nan_inf_rows':split_validation['label_null_nan_inf_rows'],
    'archive_checksums':archive_reports,
    'parquet_checksums_and_footers':dataset_identity['parquet_checksums_and_footers'],
    'test_values_read':False,'git_commit':GIT_COMMIT,'branch':BRANCH,
}
(ARTIFACT_ROOT/'dataset_identity.json').write_text(json.dumps(DATASET_IDENTITY,indent=2)+'\n',encoding='utf-8')
ENVIRONMENT.update({'run_id':RUN_ID,'git_commit':GIT_COMMIT,'branch':BRANCH,
    'dataset_manifest_sha256':transfer['dataset_manifest_sha256'],
    'source_contract_sha256':transfer['source_contract_sha256'],
    'feature_list_sha256':transfer['feature_list_sha256']})
(ARTIFACT_ROOT/'colab_environment.json').write_text(json.dumps(ENVIRONMENT,indent=2,default=str)+'\n',encoding='utf-8')
print({'run_id':RUN_ID,'dataset_root':str(DATASET_ROOT),'artifact_root':str(ARTIFACT_ROOT),'dataset_identity':DATASET_IDENTITY})


In [ ]:
import json
import ml.train_forecast_t2h_v1_1 as training

EXPECTED_CANDIDATES = {'gpu_d6_lr005','gpu_d4_regularized','gpu_d8_regularized','gpu_d6_lr003','gpu_d5_lr008'}
test_metrics_path = ARTIFACT_ROOT / 'test_metrics.json'
freeze_path = ARTIFACT_ROOT / 'freeze_manifest.json'
persistence_path = ARTIFACT_ROOT / 'persistence_validation.json'

if test_metrics_path.exists():
    test_metrics = json.loads(test_metrics_path.read_text(encoding='utf-8'))
    if not persistence_path.is_file():
        raise RuntimeError('Completed TEST report exists but persistence baseline is missing; refusing to rerun TEST')
    persistence_validation = json.loads(persistence_path.read_text(encoding='utf-8'))
    print('Completed canonical run found; reusing persisted TEST report without another TEST read.')
else:
    if freeze_path.exists():
        prior_freeze = json.loads(freeze_path.read_text(encoding='utf-8'))
        if prior_freeze.get('test_first_read_time'):
            raise RuntimeError('A TEST read is already recorded; refusing a second read')
    persistence_validation = training.write_persistence_validation(DATASET_ROOT, ARTIFACT_ROOT)
    expected = {'mae':1.453837963644283,'rmse':1.8651861753317214,'r2':0.8391473711479467,'mean_error':-0.7615142608294135}
    for key, value in expected.items():
        if abs(float(persistence_validation['metrics'][key])-value) > 0.005:
            raise RuntimeError(f'VALIDATION persistence identity mismatch for {key}')

    original_probe = training._safe_device_probe
    original_search = training.run_candidate_search
    original_fit = training.fit_final_model
    def gpu_only_probe():
        info = original_probe()
        if info.get('device') != 'cuda':
            raise RuntimeError('STOP_CANONICAL_TRAINING_NO_GPU: trainer did not select CUDA')
        return info
    def require_five_gpu_candidates(*args, **kwargs):
        records = original_search(*args, **kwargs)
        ids = {record.get('candidate_id') for record in records}
        if ids != EXPECTED_CANDIDATES or len(records) != 5 or any(r.get('status') != 'SUCCESS' for r in records):
            raise RuntimeError(f'Frozen five-candidate search did not complete: {records}')
        return records
    def require_cuda_final_fit(*args, **kwargs):
        result = original_fit(*args, **kwargs)
        if result[1] != 'cuda':
            raise RuntimeError(f'Canonical final fit fell back from CUDA to {result[1]}')
        return result
    training._safe_device_probe = gpu_only_probe
    training.run_candidate_search = require_five_gpu_candidates
    training.fit_final_model = require_cuda_final_fit
    try:
        test_metrics = training.train_and_evaluate(
            dataset_root=DATASET_ROOT, artifact_root=ARTIFACT_ROOT, model_path=MODEL_PATH,
            nthread=8, max_boost_rounds=1500, early_stopping_rounds=80,
        )
    finally:
        training._safe_device_probe = original_probe
        training.run_candidate_search = original_search
        training.fit_final_model = original_fit

candidate_results = json.loads((ARTIFACT_ROOT/'candidate_results.json').read_text(encoding='utf-8'))
winner_selection = json.loads((ARTIFACT_ROOT/'winner_selection.json').read_text(encoding='utf-8'))
freeze_manifest = json.loads(freeze_path.read_text(encoding='utf-8'))
runtime_summary = json.loads((ARTIFACT_ROOT/'runtime_summary.json').read_text(encoding='utf-8'))
reload_parity = json.loads((ARTIFACT_ROOT/'reload_parity.json').read_text(encoding='utf-8'))
model_manifest = json.loads((ARTIFACT_ROOT/'model_manifest.json').read_text(encoding='utf-8'))
per_location = json.loads((ARTIFACT_ROOT/'per_location_metrics.json').read_text(encoding='utf-8'))
assert {record['candidate_id'] for record in candidate_results} == EXPECTED_CANDIDATES
assert len(candidate_results) == 5 and all(record['status'] == 'SUCCESS' for record in candidate_results)
assert winner_selection['candidate_id'] == training.select_candidate(candidate_results)['candidate_id']
assert winner_selection['selection_metric'] == 'validation MAE' and winner_selection['test_read_before_freeze'] is False
assert freeze_manifest['configuration_frozen'] is True and freeze_manifest['test_read_before_freeze'] is False
assert freeze_manifest['freeze_time'] < freeze_manifest['test_first_read_time']
assert freeze_manifest['test_read_count'] == 1 and test_metrics['test_read_count'] == 1
assert runtime_summary['actual_candidate_device'] == 'cuda' and runtime_summary['actual_final_device'] == 'cuda'
assert test_metrics['beats_persistence_mae'] is True and test_metrics['beats_persistence_rmse'] is True
assert reload_parity['status'] == 'PASS'
assert reload_parity['max_absolute_prediction_difference'] <= reload_parity['tolerance']
assert MODEL_PATH.is_file() and MODEL_PATH.name == 'weather_forecast_xgboost_t2h_v1_1.json'
assert per_location['location_count'] == 63
print({'winner':winner_selection['candidate_id'],'freeze_time':freeze_manifest['freeze_time'],
    'test_first_read_time':freeze_manifest['test_first_read_time'],
    'test_read_count':freeze_manifest['test_read_count'],'test_metrics':test_metrics,
    'reload_parity':reload_parity})


In [ ]:
import hashlib, json
from pathlib import Path

def write_json(path, value):
    path = Path(path)
    temp = path.with_suffix(path.suffix + '.tmp')
    temp.write_text(json.dumps(value, indent=2, sort_keys=True, default=str)+'\n', encoding='utf-8')
    temp.replace(path)

local_root = REPO / 'results' / 'modeling-t2h' / LOCAL_DATASET_VERSION
local_winner = json.loads((local_root/'winner_selection.json').read_text(encoding='utf-8'))
local_test = json.loads((local_root/'test_metrics.json').read_text(encoding='utf-8'))
local_model = json.loads((local_root/'model_manifest.json').read_text(encoding='utf-8'))
local_reload = json.loads((local_root/'reload_parity.json').read_text(encoding='utf-8'))
local_locations = json.loads((local_root/'per_location_metrics.json').read_text(encoding='utf-8'))
local_validation = local_winner['validation_metrics']
colab_validation = winner_selection['validation_metrics']
colab_test = test_metrics['xgboost']
metric_pairs = {
    'validation_mae':(float(local_validation['mae']),float(colab_validation['mae'])),
    'validation_rmse':(float(local_validation['rmse']),float(colab_validation['rmse'])),
    'test_mae':(float(local_test['xgboost']['mae']),float(colab_test['mae'])),
    'test_rmse':(float(local_test['xgboost']['rmse']),float(colab_test['rmse'])),
    'test_bias':(float(local_test['xgboost']['bias']),float(colab_test['bias'])),
}
metric_deltas = {k:{'local':a,'colab':b,'absolute_delta':b-a,'relative_delta':abs(b-a)/max(abs(a),1e-12)} for k,(a,b) in metric_pairs.items()}
identity_mismatches = []
if test_metrics['feature_list_sha256'] != transfer['feature_list_sha256']: identity_mismatches.append('feature-list SHA mismatch')
if dataset_identity['total_rows'] != 3312666: identity_mismatches.append('dataset row mismatch')
if winner_selection['feature_list_sha256'] != transfer['feature_list_sha256']: identity_mismatches.append('winner feature SHA mismatch')
large_shift = any(metric_deltas[k]['relative_delta'] > 0.15 for k in ['validation_mae','validation_rmse','test_mae','test_rmse'])
colab_better = per_location['locations_better']
local_better = local_locations['locations_better']
location_regressions_vs_local = max(0, local_better - colab_better)
material_location_regressions = location_regressions_vs_local >= 10
if identity_mismatches or large_shift or material_location_regressions or not (test_metrics['beats_persistence_mae'] and test_metrics['beats_persistence_rmse']):
    comparison_status = 'MATERIAL_DIVERGENCE'
elif winner_selection['candidate_id'] == local_winner['candidate_id'] and winner_selection['best_iteration'] == local_winner['best_iteration'] and all(x['relative_delta'] <= 0.03 for x in metric_deltas.values()):
    comparison_status = 'CONSISTENT'
else:
    comparison_status = 'CONSISTENT_WITH_NUMERICAL_VARIATION'
comparison = {
    'classification':comparison_status,'protocol_commit':GIT_COMMIT,'branch':BRANCH,
    'local_run_id':LOCAL_DATASET_VERSION,'colab_run_id':RUN_ID,
    'winner':{'local':local_winner['candidate_id'],'colab':winner_selection['candidate_id']},
    'best_iteration':{'local':local_winner['best_iteration'],'colab':winner_selection['best_iteration']},
    'metrics':metric_deltas,
    'model_size_bytes':{'local':local_model['model_file_size_bytes'],'colab':MODEL_PATH.stat().st_size},
    'model_sha256':{'local':local_model['model_sha256'],'colab':hashlib.sha256(MODEL_PATH.read_bytes()).hexdigest()},
    'reload_parity':{'local':local_reload,'colab':reload_parity},
    'locations_better_than_persistence':{'local':local_better,'colab':colab_better},
    'locations_losing_vs_local':location_regressions_vs_local,
    'identity_mismatches':identity_mismatches,
    'diagnostic_thresholds':{'large_metric_relative_shift':0.15,'consistent_metric_relative_shift':0.03,'material_location_regressions_vs_local':10},
    'interpretation':'SHA differences across GPUs are expected and do not fail identity. Winner selection uses Colab VALIDATION only. This comparison is post-evaluation and does not feed model selection.',
}
write_json(ARTIFACT_ROOT/'local_vs_colab_comparison.json', comparison)
if comparison_status == 'MATERIAL_DIVERGENCE':
    raise RuntimeError(f'CANONICAL_REPRODUCTION_FAILED: investigate local-versus-Colab divergence: {comparison}')

environment_artifact = json.loads((ARTIFACT_ROOT/'colab_environment.json').read_text(encoding='utf-8'))
model_sha = hashlib.sha256(MODEL_PATH.read_bytes()).hexdigest()
model_size = MODEL_PATH.stat().st_size
drive_artifact_path = f'/content/drive/MyDrive/weather-streaming-bigdata/artifacts/weather_forecast_xgboost_t2h_v1_1/{RUN_ID}/'
canonical_manifest = {
    'canonical':True,'canonical_environment':'GOOGLE_COLAB','status':'PASS',
    'run_id':RUN_ID,'git_commit':GIT_COMMIT,'branch':BRANCH,
    'model_id':'WEATHER_XGBOOST_GLOBAL_T2H_V1_1','model_sha256':model_sha,'model_size_bytes':model_size,
    'model_filename':MODEL_PATH.name,'feature_set_id':'WEATHER_FORECAST_FE_T2H_V1_1','feature_count':73,
    'feature_list_sha256':transfer['feature_list_sha256'],'source_contract_id':'WEATHER_FORECAST_SOURCE_T2H_V1_1',
    'source_contract_sha256':transfer['source_contract_sha256'],'dataset_manifest_sha256':transfer['dataset_manifest_sha256'],
    'forecast_horizon_hours':2,'gpu_name':environment_artifact['gpu_name'],
    'train_rows':2207394,'validation_rows':553392,'test_rows':551880,'total_rows':3312666,
    'winner':winner_selection['candidate_id'],'winner_parameters':winner_selection['params'],
    'best_iteration':winner_selection['best_iteration'],'final_rounds':freeze_manifest['final_rounds'],
    'freeze_time':freeze_manifest['freeze_time'],'test_first_read_time':freeze_manifest['test_first_read_time'],
    'test_read_count':freeze_manifest['test_read_count'],'validation_metrics':winner_selection['validation_metrics'],
    'test_metrics':test_metrics['xgboost'],'persistence_test_metrics':test_metrics['persistence'],
    'test_skill_vs_persistence':test_metrics['improvement_vs_persistence'],
    'locations_better':per_location['locations_better'],'locations_equal':per_location['locations_equal'],
    'locations_worse':per_location['locations_worse'],'reload_parity':reload_parity,
    'drive_artifact_path':drive_artifact_path,'drive_model_path':drive_artifact_path+MODEL_PATH.name,
    'drive_test_predictions_path':drive_artifact_path+'test_predictions.parquet',
    'dataset_version':LOCAL_DATASET_VERSION,'dataset_drive_path':str(DRIVE_DATASET_ROOT),
    'local_vs_colab_classification':comparison_status,
}
write_json(ARTIFACT_ROOT/'canonical_model_manifest.json',canonical_manifest)
canonical_summary = {
    'status':'CANONICAL_PASS','run_id':RUN_ID,'drive_path':drive_artifact_path,
    'git_commit':GIT_COMMIT,'branch':BRANCH,'gpu':environment_artifact['gpu_name'],
    'dataset_identity':DATASET_IDENTITY,'winner':winner_selection['candidate_id'],
    'best_iteration':winner_selection['best_iteration'],'final_rounds':freeze_manifest['final_rounds'],
    'test_metrics':test_metrics['xgboost'],'persistence_test_metrics':test_metrics['persistence'],
    'model_sha256':model_sha,'model_size_bytes':model_size,'reload_parity':reload_parity,
    'local_vs_colab_comparison':comparison_status,'freeze_time':freeze_manifest['freeze_time'],
    'test_first_read_time':freeze_manifest['test_first_read_time'],'test_read_count':freeze_manifest['test_read_count'],
}
write_json(ARTIFACT_ROOT/'canonical_colab_summary.json',canonical_summary)

checksums = training.write_checksums(ARTIFACT_ROOT, MODEL_PATH)
for entry in checksums['files']:
    name = Path(entry['path']).name
    if entry.get('kind') == 'model_artifact' or name == MODEL_PATH.name:
        entry['kind'] = 'external_model_artifact'
        entry['external_path'] = canonical_manifest['drive_model_path']
    elif name == 'test_predictions.parquet':
        entry['kind'] = 'external_prediction_artifact'
        entry['external_path'] = canonical_manifest['drive_test_predictions_path']
    else:
        entry['kind'] = 'tracked_evidence_artifact'
checksums['artifact_policy'] = {
    'tracked_evidence':'Small JSON/CSV evidence is copied into Git under results/modeling-t2h/<RUN_ID>/',
    'external_model_artifact':canonical_manifest['drive_model_path'],
    'external_prediction_artifact':canonical_manifest['drive_test_predictions_path'],
    'checksums_file_excludes_itself':True,
}
write_json(ARTIFACT_ROOT/'checksums.json',checksums)
assert canonical_manifest['canonical'] is True and canonical_manifest['status'] == 'PASS'
print(json.dumps(canonical_summary,indent=2,default=str))


In [ ]:
print('CANONICAL RUN:', RUN_ID)
print('Google Drive artifacts:', drive_artifact_path)
print('Canonical model SHA-256:', canonical_manifest['model_sha256'])
print('TEST XGBoost MAE/RMSE:', test_metrics['xgboost']['mae'], test_metrics['xgboost']['rmse'])
print('TEST persistence MAE/RMSE:', test_metrics['persistence']['mae'], test_metrics['persistence']['rmse'])
print('Local vs Colab:', comparison['classification'])
assert canonical_summary['status'] == 'CANONICAL_PASS'
